# Archived local prototype — do not run

This notebook is retained for course history only. Its legacy setup loads Qwen weights into the environment running the notebook. Do not execute it on your laptop or use it for the current MeetingMind workflow. Use the local FastAPI/Next.js application with `MeetingMind_Colab_Inference.ipynb` opened in Google Colab instead.

The supported architecture keeps all LLM inference in Colab; the local backend owns transcript processing, CPU embeddings, FAISS retrieval, and source attribution.

In [ ]:
raise RuntimeError("This archived prototype is disabled. Do not install its local inference dependencies; use the Colab inference notebook instead.")


# 2 · Imports and configuration

Set the model, embedding, chunk, and retrieval limits in one place.

In [ ]:
import csv
import html
import json
import logging
import os
import re
import sys
import tempfile
import uuid
import warnings
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import faiss
import gradio as gr
import numpy as np
import torch
from langchain_classic.chains import LLMChain
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import PromptTemplate
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from youtube_transcript_api import YouTubeTranscriptApi

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
LOGGER = logging.getLogger("meetingmind")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDING_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_WORDS = 600
CHUNK_OVERLAP = 90
MAX_CHUNKS = 20
RAG_TOP_K = 4
MIN_RETRIEVAL_SCORE = 0.20

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "backend" / "app").is_dir()),
    None,
)
if PROJECT_ROOT is not None:
    sys.path.insert(0, str(PROJECT_ROOT))
    from backend.app.loaders.transcripts import extract_video_id as shared_extract_video_id
    from backend.app.loaders.transcripts import normalize_transcript as shared_normalize_transcript
    from backend.app.preprocessing.chunking import split_into_chunks as shared_split_into_chunks
else:
    shared_extract_video_id = None
    shared_normalize_transcript = None
    shared_split_into_chunks = None


# 3 · Archived local inference (disabled)

This prototype is retained for reference but its model-loading cell is disabled. Use `MeetingMind_Colab_Inference.ipynb` to run LLM inference remotely; do not download model weights to this machine.

In [ ]:
raise RuntimeError(
    "This archived prototype is disabled because it loads LLM weights locally. "
    "Use notebooks/MeetingMind_Colab_Inference.ipynb for remote inference."
)


# 4 · Transformers LLM wrapper

Wrap the local Transformers model as a LangChain LLM so the same model powers all three chains.

In [ ]:
from typing import Any, List, Optional

from langchain_core.language_models.llms import LLM
from pydantic import Field


class TransformersLLM(LLM):
    model: Any = Field(exclude=True)
    tokenizer: Any = Field(exclude=True)
    max_new_tokens: int = 512
    max_input_tokens: int = 4096

    @property
    def _llm_type(self) -> str:
        return "local-transformers-causal-lm"

    @property
    def _identifying_params(self) -> dict:
        return {"model_name": MODEL_NAME, "max_new_tokens": self.max_new_tokens}

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> str:
        _ = run_manager, kwargs
        messages = [
            {"role": "system", "content": "You are MeetingMind, an assistant for English-language meeting transcripts. Follow the user's instructions, answer in English, and never add facts that are not in the supplied transcript."},
            {"role": "user", "content": prompt},
        ]
        rendered_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = self.tokenizer(
            rendered_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_input_tokens,
        )
        input_device = self.model.get_input_embeddings().weight.device
        inputs = {key: value.to(input_device) for key, value in inputs.items()}

        with torch.inference_mode():
            generated = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        new_tokens = generated[0, inputs["input_ids"].shape[1]:]
        answer = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        if stop:
            for stop_text in stop:
                if stop_text in answer:
                    answer = answer.split(stop_text, 1)[0].rstrip()
        return answer


llm = TransformersLLM(model=model, tokenizer=tokenizer, max_new_tokens=512)
print("Local LangChain LLM wrapper is ready.")


# 5 · Input layer

Accept a YouTube URL, a pasted English transcript, or an uploaded PDF, TXT, VTT, or SRT file. YouTube captions are requested in English.

In [ ]:
def extract_video_id(url: str) -> str:
    """Extract a YouTube ID from standard, short, embed, live, or Shorts URLs."""
    if shared_extract_video_id is not None:
        return shared_extract_video_id(url)
    parsed = urlparse(url.strip())
    host = (parsed.hostname or "").lower()
    if host.startswith("www."):
        host = host[4:]

    if host == "youtu.be":
        video_id = parsed.path.lstrip("/").split("/")[0]
        if video_id:
            return video_id

    if host in ("youtube.com", "m.youtube.com", "music.youtube.com"):
        query = parse_qs(parsed.query)
        if query.get("v"):
            return query["v"][0]
        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in ("shorts", "embed", "live", "v"):
            return parts[1]

    raise ValueError("That does not look like a supported YouTube video URL.")


def fetch_youtube_transcript(url: str) -> str:
    video_id = extract_video_id(url)
    try:
        transcript = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    except Exception as error:
        raise ValueError("Captions could not be retrieved. Check that the video is public and has English captions.") from error
    text = "\n".join(item.text for item in transcript)
    return shared_normalize_transcript(text) if shared_normalize_transcript else text.strip()


def read_uploaded_transcript(file_path: str) -> str:
    path = Path(file_path)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        reader = PdfReader(str(path))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif suffix in (".txt", ".vtt", ".srt"):
        text = path.read_text(encoding="utf-8-sig", errors="replace")
        if suffix in (".vtt", ".srt"):
            timestamp_pattern = re.compile(r"^(?:\d{2}:)?\d{2}:\d{2}[,.]\d{3}\s+-->")
            if not any(timestamp_pattern.match(line.strip()) for line in text.splitlines()):
                raise ValueError(f"The {suffix[1:].upper()} file has no valid caption timestamps.")
            cleaned_lines = []
            for line in text.splitlines():
                line = line.strip()
                if not line or line.upper().startswith("WEBVTT"):
                    continue
                if re.fullmatch(r"\d+", line):
                    continue
                if "-->" in line or re.fullmatch(r"\d{2}:\d{2}(?::\d{2})?[,.]\d{3}", line):
                    continue
                cleaned_lines.append(re.sub(r"<[^>]+>", "", line))
            text = "\n".join(cleaned_lines)
    else:
        raise ValueError("Use a PDF, TXT, VTT, or SRT transcript file.")

    text = shared_normalize_transcript(text) if shared_normalize_transcript else html.unescape(text).strip()
    if not text:
        raise ValueError("No readable transcript text was found in that file.")
    return text


def resolve_meeting_input(youtube_url: str, pasted_text: str, uploaded_file: str) -> tuple[str, str]:
    if youtube_url and youtube_url.strip():
        return fetch_youtube_transcript(youtube_url), "YouTube transcript"
    if pasted_text and pasted_text.strip():
        text = shared_normalize_transcript(pasted_text) if shared_normalize_transcript else pasted_text.strip()
        return text, "Pasted transcript"
    if uploaded_file:
        return read_uploaded_transcript(uploaded_file), Path(uploaded_file).name
    raise ValueError("Add a YouTube link, paste a transcript, or upload a meeting file.")


# 6 · Chains and structured output parsers

Use StructuredOutputParser with ResponseSchema for extraction and the final report. Each prompt receives format instructions and prohibits guessing owners or deadlines.

In [ ]:
response_schemas = [
    ResponseSchema(name="Meeting Title", description="A concise title supported by the transcript. Use Not specified if unclear."),
    ResponseSchema(name="Meeting Summary", description="A concise executive summary grounded only in the provided text."),
    ResponseSchema(name="Decisions", description="List of objects with decision and context. Include only explicitly stated decisions."),
    ResponseSchema(name="Tasks", description="List of objects with task, Owner, Deadline, and Priority. Extract only explicitly stated facts. Use null for unstated Owner, Deadline, or Priority. A stated priority must be High, Medium, or Low."),
    ResponseSchema(name="Key points", description="List of important points explicitly discussed."),
    ResponseSchema(name="Open questions", description="List of unresolved questions explicitly raised."),
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
FORMAT_INSTRUCTIONS = output_parser.get_format_instructions()

chunk_prompt = PromptTemplate(
    input_variables=["chunk"],
    partial_variables={"format_instructions": FORMAT_INSTRUCTIONS},
    template=(
        "You are extracting meeting facts from one transcript segment. "
        "Extract only what is explicitly stated. Write all values in English. Do not invent owners, dates, decisions, "
        "or context. Use null for any owner, deadline, or priority not stated. "
        "Keep each field concise. "
        "Return only the requested JSON.\n\n"
        "Transcript segment:\n{chunk}\n\n{format_instructions}"
    ),
)
chunk_chain = LLMChain(llm=llm, prompt=chunk_prompt, verbose=False)

final_prompt = PromptTemplate(
    input_variables=["extractions"],
    partial_variables={"format_instructions": FORMAT_INSTRUCTIONS},
    template=(
        "Create the final meeting report from these per-segment extractions. "
        "Write in English. Use only facts present in the extractions. Do not add or infer owners, deadlines, "
        "decisions, or meeting context. Preserve Not specified values. "
        "Merge duplicates, write a useful title and executive summary, and keep task "
        "priorities as High, Medium, or Low. Return only the requested JSON.\n\n"
        "Extracted meeting facts:\n{extractions}\n\n{format_instructions}"
    ),
)
final_chain = LLMChain(llm=llm, prompt=final_prompt, verbose=False)

qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Answer the question using only the meeting transcript excerpts below. "
        "Do not use outside knowledge or infer missing facts. If the excerpts do not "
        "contain the answer, say exactly: I couldn't find this information in the meeting. "
        "Be concise and answer in English.\n\n"
        "Retrieved transcript excerpts:\n{context}\n\nQuestion: {question}\nAnswer:"
    ),
)
qa_chain = LLMChain(llm=llm, prompt=qa_prompt, verbose=False)


def _json_candidate(text: str) -> str:
    fenced = re.search(r"\x60{3}(?:json)?\s*(.*?)\s*\x60{3}", text, flags=re.IGNORECASE | re.DOTALL)
    candidate_text = fenced.group(1) if fenced else text
    start = candidate_text.find("{")
    if start < 0:
        raise ValueError("The model response did not contain a JSON object.")
    decoder = json.JSONDecoder()
    _, end = decoder.raw_decode(candidate_text[start:])
    return candidate_text[start:start + end]


def safe_parse(raw_text: str) -> dict:
    try:
        parsed = output_parser.parse(raw_text)
    except OutputParserException as parser_error:
        try:
            parsed = json.loads(_json_candidate(raw_text))
        except (ValueError, json.JSONDecodeError) as json_error:
            raise ValueError(f"Could not parse the model response as structured JSON: {json_error}") from parser_error
        warnings.warn("StructuredOutputParser format was skipped; parsed the JSON response with the safe fallback.", stacklevel=2)

    if not isinstance(parsed, dict):
        raise ValueError("The parsed meeting response must be a JSON object.")
    for key, default in (
        ("Meeting Title", "Not specified"),
        ("Meeting Summary", ""),
        ("Decisions", []),
        ("Tasks", []),
        ("Key points", []),
        ("Open questions", []),
    ):
        parsed.setdefault(key, default)
    return parsed


# 7 · Pipeline functions

Split long transcripts with overlap, cap processing at 20 chunks, extract each chunk, merge duplicates, generate the final report, and build a FAISS index over source chunks.

In [ ]:
def split_into_chunks(text: str, chunk_words: int = CHUNK_WORDS, overlap: int = CHUNK_OVERLAP) -> tuple[list[str], bool]:
    if shared_split_into_chunks is not None:
        chunks, truncated = shared_split_into_chunks(text, chunk_words, overlap, MAX_CHUNKS)
        return [chunk.text for chunk in chunks], truncated
    words = text.split()
    if not words:
        raise ValueError("The transcript is empty.")
    step = max(1, chunk_words - overlap)
    all_chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            all_chunks.append(chunk)
        if end == len(words):
            break
        start += step
    truncated = len(all_chunks) > MAX_CHUNKS
    return all_chunks[:MAX_CHUNKS], truncated


def _field_text(item: Any, key: str) -> str:
    if isinstance(item, dict):
        return str(item.get(key, "")).strip()
    return str(item).strip()


def _dedupe_records(records: list, identity_key: Optional[str] = None) -> list:
    unique = []
    positions = {}
    for record in records:
        if isinstance(record, dict):
            identity = _field_text(record, identity_key or "decision").casefold()
            if not identity:
                continue
            if identity in positions:
                prior = unique[positions[identity]]
                for key, value in record.items():
                    old_value = str(prior.get(key, "")).strip()
                    new_value = str(value).strip()
                    if new_value and (not old_value or old_value.casefold() == "not specified"):
                        prior[key] = value
                continue
            positions[identity] = len(unique)
            unique.append(dict(record))
        else:
            value = str(record).strip()
            identity = value.casefold()
            if value and identity not in positions:
                positions[identity] = len(unique)
                unique.append(value)
    return unique


def merge_extractions(extractions: list[dict]) -> dict:
    merged = {
        "Meeting Title": "Not specified",
        "Meeting Summary": "",
        "Decisions": [],
        "Tasks": [],
        "Key points": [],
        "Open questions": [],
    }
    for item in extractions:
        if merged["Meeting Title"] == "Not specified" and item.get("Meeting Title") not in (None, "", "Not specified"):
            merged["Meeting Title"] = item["Meeting Title"]
        if item.get("Meeting Summary"):
            merged["Meeting Summary"] += str(item["Meeting Summary"]).strip() + "\n"
        for key in ("Decisions", "Tasks", "Key points", "Open questions"):
            values = item.get(key, [])
            if isinstance(values, list):
                merged[key].extend(values)

    merged["Meeting Summary"] = merged["Meeting Summary"].strip()
    merged["Decisions"] = _dedupe_records(merged["Decisions"], "decision")
    merged["Tasks"] = _dedupe_records(merged["Tasks"], "task")
    merged["Key points"] = _dedupe_records(merged["Key points"])
    merged["Open questions"] = _dedupe_records(merged["Open questions"])
    return merged


def normalize_report(report: dict) -> dict:
    normalized = dict(report)
    normalized["Meeting Title"] = str(normalized.get("Meeting Title") or "Not specified").strip()
    normalized["Meeting Summary"] = str(normalized.get("Meeting Summary") or "").strip()
    decisions = []
    for item in _items(normalized.get("Decisions")):
        if isinstance(item, dict):
            decision = str(item.get("decision") or "").strip()
            context = str(item.get("context") or "").strip()
        else:
            decision, context = str(item).strip(), ""
        if decision:
            decisions.append({"decision": decision, "context": context})
    normalized["Decisions"] = _dedupe_records(decisions, "decision")
    tasks = []
    for item in _items(normalized.get("Tasks")):
        task = item if isinstance(item, dict) else {"task": str(item)}
        priority = str(task.get("Priority") or "").title()
        if priority not in ("High", "Medium", "Low"):
            priority = None
        owner = str(task.get("Owner") or "").strip()
        if owner.casefold() in ("", "not specified", "unknown", "n/a"):
            owner = None
        deadline = str(task.get("Deadline") or "").strip()
        if deadline.casefold() in ("", "not specified", "unknown", "n/a"):
            deadline = None
        task_text = str(task.get("task") or "").strip()
        if task_text:
            tasks.append({
                "task": task_text,
                "Owner": owner,
                "Deadline": deadline,
                "Priority": priority,
            })
    normalized["Tasks"] = _dedupe_records(tasks, "task")
    for key in ("Key points", "Open questions"):
        values = _items(normalized.get(key))
        normalized[key] = _dedupe_records(values)
    return normalized


def build_faiss_index(chunks: list[str]) -> tuple[Any, np.ndarray]:
    vectors = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index, vectors


def run_meeting_pipeline(text: str, source_name: str, progress=gr.Progress()) -> dict:
    chunks, truncated = split_into_chunks(text)
    extracted = []
    for number, chunk in enumerate(chunks, start=1):
        progress((number - 1) / (len(chunks) + 2), desc=f"Extracting segment {number} of {len(chunks)}")
        raw_result = chunk_chain.invoke({"chunk": chunk})["text"]
        extracted.append(safe_parse(raw_result))

    progress(len(chunks) / (len(chunks) + 2), desc="Merging meeting notes")
    merged = merge_extractions(extracted)
    compact_extractions = json.dumps(merged, ensure_ascii=False)
    raw_final = final_chain.invoke({"extractions": compact_extractions})["text"]
    report = normalize_report(safe_parse(raw_final))

    index, _ = build_faiss_index(chunks)
    progress(1.0, desc="Meeting analysis complete")
    return {
        "report": report,
        "chunks": chunks,
        "index": index,
        "source_name": source_name,
        "word_count": len(text.split()),
        "truncated": truncated,
    }


def answer_meeting_question(question: str, meeting_state: dict) -> tuple[str, list[tuple[int, str]]]:
    if not question.strip():
        raise ValueError("Enter a question about the meeting.")
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    scores, indexes = meeting_state["index"].search(query_vector, min(RAG_TOP_K, len(meeting_state["chunks"])))
    if len(indexes[0]) == 0 or indexes[0][0] < 0 or float(scores[0][0]) < MIN_RETRIEVAL_SCORE:
        return "I couldn't find this information in the meeting.", []
    sources = [
        (int(position) + 1, meeting_state["chunks"][int(position)])
        for position in indexes[0]
        if position >= 0
    ]
    context = "\n\n".join(f"[Segment {number}]\n{text}" for number, text in sources)
    answer = qa_chain.invoke({"context": context, "question": question.strip()})["text"].strip()
    return answer, sources


# 8 · UI helpers: HTML and CSS

Render responsive summary cards and decision/action-item views. Escape transcript-derived text before inserting it into HTML.

In [ ]:
CSS = """
:root { --mm-ink: #e9efff; --mm-muted: #9eaccb; --mm-card: rgba(255,255,255,.055); }
.gradio-container { max-width: 1440px !important; margin: 0 auto !important; background: transparent !important; }
body { background: radial-gradient(ellipse at 8% 0%, rgba(80,76,220,.19), transparent 35%), #0b1020 !important; }
.mm-hero { padding: 30px 34px; border: 1px solid rgba(160,170,255,.19); border-radius: 24px; background: linear-gradient(115deg,rgba(46,49,125,.92),rgba(16,24,49,.96) 60%,rgba(19,96,112,.72)); box-shadow: 0 20px 60px rgba(0,0,0,.24); color: var(--mm-ink); }
.mm-brand { color:#b7c3ff; font-size:12px; font-weight:800; letter-spacing:.18em; text-transform:uppercase; }
.mm-hero h1 { margin:10px 0 8px; font-size:clamp(30px,4vw,48px); line-height:1.05; letter-spacing:-.04em; }
.mm-hero p { margin:0; max-width:760px; color:#c2cbe0; font-size:15px; }
.mm-pills { display:flex; flex-wrap:wrap; gap:8px; margin-top:20px; }
.mm-pill { border:1px solid rgba(215,222,255,.18); border-radius:99px; padding:6px 11px; color:#e8eaff; background:rgba(255,255,255,.07); font-size:12px; }
.mm-panel, .mm-card { border:1px solid rgba(160,170,205,.13); border-radius:18px; background:var(--mm-card); padding:18px; color:var(--mm-ink); }
.mm-kpis { display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:12px; margin:14px 0; }
.mm-kpi { border:1px solid rgba(160,170,205,.13); border-radius:15px; padding:16px; background:rgba(255,255,255,.045); }
.mm-kpi strong { display:block; font-size:25px; color:#f2f4ff; }
.mm-kpi span, .mm-label { color:var(--mm-muted); font-size:12px; }
.mm-title { margin:20px 0 10px; color:#f2f4ff; font-size:16px; font-weight:750; }
.mm-list { display:grid; gap:10px; }
.mm-card h3 { margin:0 0 6px; font-size:15px; }
.mm-card p { margin:0; color:#b8c3da; line-height:1.55; white-space:pre-wrap; }
.mm-table-wrap { overflow-x:auto; border:1px solid rgba(160,170,205,.13); border-radius:14px; }
.mm-table { width:100%; border-collapse:collapse; min-width:620px; color:#e9efff; }
.mm-table th,.mm-table td { padding:12px 14px; text-align:left; border-bottom:1px solid rgba(160,170,205,.12); vertical-align:top; }
.mm-table th { color:#abb9d6; font-size:12px; text-transform:uppercase; letter-spacing:.06em; }
.mm-badge { display:inline-block; border-radius:99px; padding:4px 9px; font-size:11px; font-weight:800; }
.mm-high { color:#ffd2d2; background:rgba(248,90,102,.18); }
.mm-medium { color:#ffe2ab; background:rgba(245,174,57,.17); }
.mm-low { color:#b8f4db; background:rgba(49,199,137,.17); }
.mm-unset { color:#c1c9d8; background:rgba(160,170,190,.12); }
.mm-empty { border:1px dashed rgba(160,170,205,.25); border-radius:14px; padding:22px; color:#aab6cf; text-align:center; }
.mm-summary { color:#d4def1; line-height:1.7; white-space:pre-wrap; }
.mm-source { border-left:2px solid #8589ff; padding:9px 12px; margin:8px 0; color:#bec9df; background:rgba(255,255,255,.035); border-radius:0 10px 10px 0; white-space:pre-wrap; }
.mm-primary button { min-height:52px; font-weight:800 !important; border-radius:14px !important; }
@media(max-width:760px){ .mm-kpis{grid-template-columns:repeat(2,minmax(0,1fr));}.mm-hero{padding:23px 20px;} }
"""


def _safe(value: Any) -> str:
    return html.escape(str(value or ""), quote=True)


def _items(value: Any) -> list:
    return value if isinstance(value, list) else []


def _empty(message: str) -> str:
    return f'<div class="mm-empty">{_safe(message)}</div>'


def render_overview(report: dict, words: int, minutes: int) -> str:
    decisions = _items(report.get("Decisions"))
    tasks = _items(report.get("Tasks"))
    summary = _safe(report.get("Meeting Summary", ""))
    title = _safe(report.get("Meeting Title", "Meeting overview"))
    key_points = _items(report.get("Key points"))
    questions = _items(report.get("Open questions"))
    html_parts = [
        f'<div class="mm-card"><div class="mm-label">MEETING BRIEF</div><h2>{title}</h2><div class="mm-summary">{summary or "No summary was returned."}</div></div>',
        f'<div class="mm-kpis"><div class="mm-kpi"><strong>{len(tasks)}</strong><span>Action items</span></div><div class="mm-kpi"><strong>{len(decisions)}</strong><span>Decisions</span></div><div class="mm-kpi"><strong>{words:,}</strong><span>Transcript words</span></div><div class="mm-kpi"><strong>{minutes}</strong><span>Est. minutes</span></div></div>',
    ]
    for heading, values in (("Key points", key_points), ("Open questions", questions)):
        if values:
            cards = "".join(f'<div class="mm-card"><p>{_safe(_field_text(value, "point") if isinstance(value, dict) else value)}</p></div>' for value in values)
            html_parts.append(f'<div class="mm-title">{_safe(heading)}</div><div class="mm-list">{cards}</div>')
        else:
            html_parts.append(f'<div class="mm-title">{_safe(heading)}</div>{_empty("Nothing was explicitly captured.")}')
    return "".join(html_parts)


def render_decisions(report: dict) -> str:
    decisions = _items(report.get("Decisions"))
    if not decisions:
        return _empty("No explicit decisions were found in the meeting.")
    cards = []
    for item in decisions:
        decision = _field_text(item, "decision")
        context = _field_text(item, "context")
        cards.append(f'<div class="mm-card"><h3>{_safe(decision or "Decision")}</h3><p>{_safe(context or "No additional context stated.")}</p></div>')
    return f'<div class="mm-list">{"".join(cards)}</div>'


def render_actions(report: dict) -> str:
    tasks = _items(report.get("Tasks"))
    if not tasks:
        return _empty("No explicit action items were found in the meeting.")
    rows = []
    for item in tasks:
        if not isinstance(item, dict):
            item = {"task": str(item)}
        priority = str(item.get("Priority") or "Not specified").title()
        badge_class = {"High": "mm-high", "Medium": "mm-medium", "Low": "mm-low"}.get(priority, "mm-unset")
        rows.append(
            "<tr>"
            f"<td>{_safe(item.get('task', ''))}</td>"
            f"<td>{_safe(item.get('Owner', 'Not specified'))}</td>"
            f"<td>{_safe(item.get('Deadline', 'Not specified'))}</td>"
            f'<td><span class="mm-badge {badge_class}">{_safe(priority)}</span></td>'
            "</tr>"
        )
    return (
        '<div class="mm-table-wrap"><table class="mm-table"><thead><tr>'
        "<th>Task</th><th>Owner</th><th>Deadline</th><th>Priority</th>"
        f"</tr></thead><tbody>{''.join(rows)}</tbody></table></div>"
    )


def render_sources(sources: list[tuple[int, str]]) -> str:
    if not sources:
        return _empty("No transcript sources are available.")
    return "".join(
        f'<div class="mm-source"><strong>Segment {number}</strong><br>{_safe(text)}</div>'
        for number, text in sources
    )


def write_downloads(report: dict) -> tuple[str, str]:
    report_id = uuid.uuid4().hex[:10]
    csv_path = os.path.join(tempfile.gettempdir(), f"meetingmind-actions-{report_id}.csv")
    markdown_path = os.path.join(tempfile.gettempdir(), f"meetingmind-notes-{report_id}.md")
    tasks = _items(report.get("Tasks"))
    with open(csv_path, "w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["task", "Owner", "Deadline", "Priority"])
        writer.writeheader()
        for item in tasks:
            if isinstance(item, dict):
                writer.writerow({key: item.get(key, "") for key in writer.fieldnames})
            else:
                writer.writerow({"task": str(item), "Owner": "Not specified", "Deadline": "Not specified", "Priority": "Low"})

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write(f"# {report.get('Meeting Title', 'Meeting notes')}\n\n")
        file.write(f"## Meeting Summary\n\n{report.get('Meeting Summary', '')}\n\n")
        for heading, key in (("Decisions", "Decisions"), ("Key points", "Key points"), ("Open questions", "Open questions")):
            file.write(f"## {heading}\n\n")
            for item in _items(report.get(key)):
                if isinstance(item, dict):
                    text = f"**{item.get('decision', item.get('point', ''))}** — {item.get('context', '')}"
                else:
                    text = str(item)
                file.write(f"- {text}\n")
            file.write("\n")
        file.write("## Action items\n\n| Task | Owner | Deadline | Priority |\n|---|---|---|---|\n")
        for item in tasks:
            if not isinstance(item, dict):
                item = {"task": str(item)}
            cells = [str(item.get(key, "Not specified")).replace("|", "\\|") for key in ("task", "Owner", "Deadline", "Priority")]
            file.write("| " + " | ".join(cells) + " |\n")
    return csv_path, markdown_path


# 9 · Gradio handlers

Connect the inputs, analysis pipeline, dashboard views, and grounded Q&A to Gradio components.

In [ ]:
SAMPLE_MEETING = """November product launch planning — weekly meeting
Participants: Maya (Product), Omar (Engineering), Priya (Design), and Nabil (Operations).
Maya: We agreed to launch the refreshed onboarding flow on 14 November, provided QA signs off.
Omar: I will deliver the final onboarding copy by 7 November. This is a high-priority task.
Maya: Priya will run the accessibility review by 10 November.
Omar: The team decided to keep the existing pricing page for launch and revisit it after the first month.
Priya: We need to obtain legal approval for the launch copy, but no owner has been assigned.
Nabil: I will confirm support coverage for launch weekend by 12 November.
Maya: We decided to use the current analytics dashboard for launch and revisit extra metrics after the first week.
Maya: The signup completion rate improved by 8% in the latest usability test.
Omar: I will send the launch checklist to the team tomorrow.
Priya: Will customer support be available throughout launch weekend?
"""


def analyze_meeting(youtube_url: str, pasted_text: str, uploaded_file: str, progress=gr.Progress()):
    try:
        transcript, source_name = resolve_meeting_input(youtube_url, pasted_text, uploaded_file)
        if len(transcript.split()) < 8:
            raise ValueError("Please provide a little more transcript text (at least 8 words).")
        meeting_state = run_meeting_pipeline(transcript, source_name, progress)
        report = meeting_state["report"]
        words = meeting_state["word_count"]
        minutes = max(1, round(words / 130))
        overview_html = render_overview(report, words, minutes)
        decision_html = render_decisions(report)
        action_html = render_actions(report)
        csv_path, markdown_path = write_downloads(report)
        notice = f"Analysis complete · {source_name} · {words:,} words"
        if meeting_state["truncated"]:
            notice += f" · Long transcript: processed the first {MAX_CHUNKS} overlapping segments."
            gr.Warning("This transcript exceeded the processing limit. The first 20 segments were analyzed.")
        gr.Info(notice)
        return notice, overview_html, decision_html, action_html, csv_path, markdown_path, meeting_state
    except gr.Error:
        raise
    except Exception as error:
        LOGGER.exception("Meeting analysis failed")
        raise gr.Error(f"Could not analyse this meeting: {error}") from error


def load_sample():
    return "", SAMPLE_MEETING, None, "Sample meeting loaded. Select Analyse meeting to run the local model."


def ask_meeting(question: str, history: list, meeting_state: dict):
    history = history or []
    if not meeting_state:
        gr.Warning("Analyse a meeting first, then ask a question about it.")
        return history, _empty("Analyse a meeting to enable source citations.")
    try:
        answer, sources = answer_meeting_question(question or "", meeting_state)
        history = history + [
            {"role": "user", "content": question.strip()},
            {"role": "assistant", "content": answer},
        ]
        return history, render_sources(sources)
    except Exception as error:
        LOGGER.exception("Meeting Q&A failed")
        raise gr.Error(f"Could not answer this question: {error}") from error


# 10 · Launch

Launch the full Gradio app. In Docker, open JupyterLab on port 8888 and run the notebook cells in order; the app is served on port 7860 with a temporary Gradio share link.

In [ ]:
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="indigo", neutral_hue="slate"),
    css=CSS,
    title="MeetingMind — AI Meeting Intelligence",
) as demo:
    gr.HTML(
        """
        <section class="mm-hero">
          <div class="mm-brand">MeetingMind · AI meeting intelligence</div>
          <h1>Turn conversations into clarity.</h1>
          <p>Grounded summaries, accountable action items, and transcript-based answers — all processed with a local open model.</p>
          <div class="mm-pills">
            <span class="mm-pill">Qwen 2.5 1.5B · 4-bit</span>
            <span class="mm-pill">FAISS retrieval</span>
            <span class="mm-pill">English-first</span>
            <span class="mm-pill">Local-first analysis</span>
          </div>
        </section>
        """
    )

    meeting_state = gr.State(value=None)
    with gr.Row(equal_height=False):
        with gr.Column(scale=9, min_width=310):
            gr.Markdown("### Add a meeting")
            with gr.Tabs():
                with gr.Tab("YouTube"):
                    youtube_input = gr.Textbox(
                        label="YouTube link",
                        placeholder="https://www.youtube.com/watch?v=…",
                        lines=2,
                    )
                    gr.Markdown("English captions are requested.")
                with gr.Tab("Paste text"):
                    transcript_input = gr.Textbox(
                        label="Meeting transcript",
                        placeholder="Paste your transcript here…",
                        lines=12,
                    )
                with gr.Tab("Upload"):
                    upload_input = gr.File(
                        label="Transcript or meeting PDF",
                        file_types=[".pdf", ".txt", ".vtt", ".srt"],
                        type="filepath",
                    )
            analyze_button = gr.Button("Analyse meeting", variant="primary", elem_classes=["mm-primary"])
            sample_button = gr.Button("Load sample meeting", variant="secondary")
            status_message = gr.Markdown("Ready when you are. Your meeting text stays in this session.")

        with gr.Column(scale=15, min_width=380):
            with gr.Tabs():
                with gr.Tab("Overview"):
                    overview_view = gr.HTML(_empty("Add a meeting and select Analyse meeting to get started."))
                with gr.Tab("Decisions"):
                    decisions_view = gr.HTML(_empty("Your explicit meeting decisions will appear here."))
                with gr.Tab("Action items"):
                    actions_view = gr.HTML(_empty("Action items with owners, deadlines, and priorities will appear here."))
                    with gr.Row():
                        csv_download = gr.DownloadButton("Download CSV", interactive=False)
                        markdown_download = gr.DownloadButton("Download meeting notes", interactive=False)
                with gr.Tab("Ask the meeting"):
                    gr.Markdown("Ask about the transcript. Answers use retrieved meeting excerpts only.")
                    chat_view = gr.Chatbot(type="messages", height=380, show_copy_button=True, label="Meeting Q&A")
                    with gr.Row():
                        question_input = gr.Textbox(
                            label="Question",
                            placeholder="What did the team decide about the launch?",
                            scale=6,
                            lines=1,
                        )
                        ask_button = gr.Button("Ask", variant="primary", scale=1)
                    gr.Examples(
                        examples=[
                            "What launch date was agreed?",
                            "Who owns the final onboarding copy?",
                            "What is still an open question?",
                        ],
                        inputs=question_input,
                    )
                    with gr.Accordion("Retrieved transcript sources", open=False):
                        sources_view = gr.HTML(_empty("Sources appear here after a question."))

    analyze_button.click(
        fn=analyze_meeting,
        inputs=[youtube_input, transcript_input, upload_input],
        outputs=[status_message, overview_view, decisions_view, actions_view, csv_download, markdown_download, meeting_state],
        show_progress="full",
    )
    youtube_input.input(
        fn=lambda _: ("", None),
        inputs=youtube_input,
        outputs=[transcript_input, upload_input],
    )
    transcript_input.input(
        fn=lambda _: ("", None),
        inputs=transcript_input,
        outputs=[youtube_input, upload_input],
    )
    upload_input.upload(
        fn=lambda _: ("", ""),
        inputs=upload_input,
        outputs=[youtube_input, transcript_input],
    )
    sample_button.click(
        fn=load_sample,
        inputs=[],
        outputs=[youtube_input, transcript_input, upload_input, status_message],
    )
    ask_button.click(
        fn=ask_meeting,
        inputs=[question_input, chat_view, meeting_state],
        outputs=[chat_view, sources_view],
    )
    question_input.submit(
        fn=ask_meeting,
        inputs=[question_input, chat_view, meeting_state],
        outputs=[chat_view, sources_view],
    )

demo.queue(default_concurrency_limit=1).launch(
    share=True,
    server_name="0.0.0.0",
    server_port=7860,
    show_error=True,
)


# 11 · Demo checklist

- Choose a GPU runtime (T4 recommended) and run every cell from top to bottom.
- Click Load sample meeting, then Analyse meeting.
- Review the overview, decisions, and action items; download CSV or Markdown notes.
- Ask who owns the onboarding copy and inspect the retrieved transcript sources.
- Try a YouTube link with English captions or upload a PDF, TXT, VTT, or SRT transcript in English.

- Docker quick start: build with docker build -t meetingmind .; then run with docker run --gpus all --rm -p 8888:8888 -p 7860:7860 -e JUPYTER_TOKEN=use-a-long-random-token meetingmind. Open port 8888, run this notebook, and use the Gradio link.

Five-line presentation script:
1. MeetingMind turns a transcript, YouTube captions, or a meeting file into structured notes.
2. The overview highlights the summary, decisions, action items, transcript size, and duration.
3. Action items show only stated owners and deadlines, and can be downloaded as CSV or Markdown.
4. Ask the meeting uses FAISS to retrieve source segments before the local model answers.
5. Open the source panel to verify the evidence behind an answer.

In [ ]:
assert MODEL_NAME == "Qwen/Qwen2.5-1.5B-Instruct"
assert EMBEDDING_NAME == "sentence-transformers/all-MiniLM-L6-v2"
assert output_parser is not None and chunk_chain is not None and final_chain is not None and qa_chain is not None
assert callable(extract_video_id) and callable(run_meeting_pipeline) and callable(answer_meeting_question)
print("Demo checklist ready. Start with Load sample meeting.")
